In [ ]:
!pip install -U "transformers>=4.42.0" "accelerate>=0.32.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" "trl>=0.8.6" "wandb" "pandera" -q

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
import wandb

user_secrets = UserSecretsClient()

os.environ["HF_TOKEN"]  = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

PROJECT_NAME = "nemo-text-to-sql-phase2"
HUB_REPO_ID = f"NBAmine/{PROJECT_NAME}"

PHASE1_REPO_ID = "NBAmine/nemo-text-to-sql-phase1"

wandb.login()

In [ ]:
from datasets import load_dataset

def process_spider(sample):
    return {
        "text": (
            f"Context: {sample['db_id']}\n"
            f"Question: {sample['question']}\n"
            f"Answer: {sample['query']}"
        )
    }

ds_spider_train = load_dataset("xlangai/spider", split="train")
ds_spider_train = ds_spider_train.map(lambda x: process_spider(x))

ds_spider_val = load_dataset("xlangai/spider", split="validation")
ds_spider_val = ds_spider_val.map(lambda x: process_spider(x))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, PeftModel
import torch

model_id = "mistralai/Mistral-Nemo-Instruct-2407"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    use_cache=False,
    attn_implementation="sdpa"
)

base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

base_model = prepare_model_for_kbit_training(base_model)

print(f"Loading Phase 1 adapters from {PHASE1_REPO_ID}...")

model = PeftModel.from_pretrained(
    base_model, 
    PHASE1_REPO_ID,
    is_trainable=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    fix_mistral_regex=True,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"

In [ ]:
import numpy as np

def compute_length(batch):
    return {"len": len(tokenizer.encode(batch["text"]))}

print("Processing training set lengths...")
ds_train_with_len = ds_spider_train.map(compute_length)

print("Processing validation set lengths...")
ds_val_with_len = ds_spider_train.map(compute_length)

train_lengths = ds_train_with_len["len"]
val_lengths = ds_val_with_len["len"]

max_train = max(train_lengths)
max_val = max(val_lengths)

p95_train = np.percentile(train_lengths, 95)
p99_train = np.percentile(train_lengths, 99)

print("-" * 30)
print(f"  MAX Token Length (Train): {max_train}")
print(f"  MAX Token Length (Test):  {max_val}")
print("-" * 30)
print(f"  95% of train data is under:     {p95_train:.0f} tokens")
print(f"  99% of train data is under:     {p99_train:.0f} tokens")
print("-" * 30)

MAX_LENGTH = 1024

tokenizer.model_max_length = MAX_LENGTH

if max_train > 2000:
    print("WARNING: We have massive outliers.")
    print(f"MAX_LENGTH set to {MAX_LENGTH}. We let the Trainer truncate the outliers.") 
else:
    print(f"MAX_LENGTH of tokenizer set to {MAX_LENGTH}.") 

In [ ]:
from transformers import TrainerCallback
import torch

class SQLSampleCallback(TrainerCallback):

    def __init__(self, eval_dataset, tokenizer, model):
        self.model = model
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.samples = self.eval_dataset.shuffle(seed=42).select(range(8))

    def on_evaluate(self, args, state, control, **kwargs):

        epoch = state.epoch if state.epoch is not None else -1
        global_step = state.global_step
        device = self.model.device

        # average performance of the model during the period since the last N logging_steps
        train_loss = 0.0
        if state.log_history:
            for log in reversed(state.log_history):
                if "loss" in log:
                    train_loss = log["loss"]
                    break

        columns = ["Epoch", "Step", "Train Loss", "Context", "Question", "True SQL", "Generated SQL"]
        data = []

        self.model.eval()

        for sample in self.samples:

            text = sample['text']

            context = text.split("Question:")[0].replace("Context: ", "").strip()
            question = text.split("Question:")[1].split("Answer:")[0].strip()
            true_sql = text.split("Answer:")[1].strip()

            inputs = self.tokenizer(f"Context: {context}\nQuestion: {question}\nAnswer:", return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = self.model.generate(**inputs, max_new_tokens=100, pad_token_id=self.tokenizer.unk_token_id)

            gen_sql = self.tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1].strip()

            data.append([epoch, global_step, train_loss, context, question, true_sql, gen_sql])

        self.model.train()
        
        wandb.log({"validation_samples": wandb.Table(data=data, columns=columns)})

In [ ]:
import numpy as np

# 1. Preprocess: Runs on GPU. Reduces huge logits to just the "predicted token ID"
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

# 2. Compute: Runs on CPU. Compares predictions vs labels.
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    
    # The labels have -100 for ignored tokens (padding/user prompt). 
    # We only want to calculate accuracy on the "completion" (the SQL part).
    mask = labels != -100
    
    # Filter arrays to only include relevant tokens
    relevant_preds = preds[mask]
    relevant_labels = labels[mask]
    
    # Calculate accuracy
    accuracy = (relevant_preds == relevant_labels).mean()
    
    return {"accuracy": accuracy}

In [ ]:
from pathlib import Path
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import HfApi, snapshot_download

def run_training(run_name, epochs, output_dir, train_dataset, eval_dataset, learning_rate):

    last_checkpoint=None
    output_path = Path(output_dir)
    
    if output_path.exists() and output_path.is_dir():
        last_checkpoint = get_last_checkpoint(output_dir)
        print(f"Found checkpoint: {last_checkpoint}")
        print(f"Resuming training from {last_checkpoint}...")
        
    else :
        try:
            api = HfApi()

            if api.repo_exists(repo_id=HUB_REPO_ID) and "last-checkpoint/trainer_state.json" in api.list_repo_files(repo_id=HUB_REPO_ID):
                
                print("Downloading checkpoint from Cloud ...")
                
                download_path = snapshot_download(
                    repo_id=HUB_REPO_ID,
                    allow_patterns=["last-checkpoint/*"]
                )
                    
                last_checkpoint = os.path.join(download_path, "last-checkpoint")
                print(f"Downloaded to: {last_checkpoint}")
            else:
                print("No Hub checkpoint found. Starting fresh training...")
                
        except Exception as e:
            print(f"Could not check Hub status: {e}. Starting fresh.")

    if wandb.run is not None:
        wandb.finish()
        
    wandb.init(
        project=PROJECT_NAME,
        name=run_name,
        id=run_name,
        resume="allow",
        tags=["phase2", "logical-alignement", "mistral-nemo", "qlora"],
        notes="Training on 7k xlangai/spider samples for logic injection.",
        config={
            "dataset_name": "xlangai/spider",
            "dataset_size": "7k",
            "model_type": model_id,
        }
    )

    args = SFTConfig(
        output_dir = output_dir,
        num_train_epochs = epochs,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate = learning_rate,
        
        fp16 = True,
        bf16 = False,
        logging_steps = 10,
        ddp_find_unused_parameters=False,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        
        optim = "paged_adamw_8bit",
        report_to = "wandb",
        group_by_length = True,
        load_best_model_at_end=True,

        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        per_device_eval_batch_size=4,
        
        push_to_hub=True,                   
        hub_model_id=HUB_REPO_ID,
        hub_strategy="checkpoint",
        hub_token=os.environ["HF_TOKEN"]
    )

    sql_sample_callback = SQLSampleCallback(
        model=model,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=args,
        processing_class=tokenizer,
        callbacks=[sql_sample_callback],

        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        compute_metrics=compute_metrics,
    )

    trainer.model.print_trainable_parameters()
    
    print("Explicitely converting from bfloat16 to float32...")
    
    for name, param in trainer.model.named_parameters():
        
        # Fix LoRA weights
        if "lora" in name and param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float32)
        
         # Fix any other stray bfloat16 params (like LayerNorm)
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
            
    print("All adapters are now Float32.")
        
    trainer.train(resume_from_checkpoint=last_checkpoint)

    trainer.model.save_pretrained(output_dir)

    wandb.finish()

In [ ]:
run_training(
    run_name="phase2-logical-alignment",
    train_dataset=ds_spider_train,
    eval_dataset=ds_spider_val,
    epochs=5,
    output_dir="./adapter-phase2",
    learning_rate=1e-5
)